In [ ]:
import re
import sys
from pathlib import Path as ph
import matplotlib.pyplot as plt

In [ ]:
# Add parent directory to sys.path
sys.path.append(str(ph().resolve().parent))
from src.functions.runtime import from_file, towa_file, get_directory_files

In [ ]:
runtime_path_inp = input("Enter the runtime path ('same','<path>'): ").strip().lower()
runtime_uuid_inp = input("Enter the model configuration ('<uuid>','all'): ").strip().lower()

In [ ]:
########################
# Runtime variables
########################

if runtime_path_inp == "same":
    runtime_path = "."
else:
    runtime_path = runtime_path_inp

runtime_uuid = runtime_uuid_inp

configs_path = f"{runtime_path}/configs"
tokenizers_path = f"{runtime_path}/tokenizers"
inputs_path = f"{runtime_path}/inputs"
outputs_path = f"{runtime_path}/outputs"
models_path = f"{runtime_path}/models"
charts_path = f"{runtime_path}/charts"
statistics_path = f"{runtime_path}/statistics"

In [ ]:
########################
# Visualize file components
########################

def process_chart_file_components(folder_path, file_prefix, runtime_uuid):
    runtime_uuids = []
    if runtime_uuid == "all":
        # If a all is provided, add ithem all to the list
        runtime_uuids = get_directory_files(folder_path, file_prefix)
    else:
        # If a specific UUID is provided, add it to the list
        runtime_uuids.append(runtime_uuid)

    # Loop through each GUID to extract weights
    for runtime_uuid in runtime_uuids:
        print(runtime_uuid)
        # Load the model from a single file
        report_path_inp = f"{outputs_path}/report_{runtime_uuid}.json"
        report = from_file(report_path_inp, "json")

        batches_loss = []
        for line in report["batch_logs"]:
            if "train_loss" in line:
                match = re.search(r"train_loss\s*=\s*([0-9.]+)", line)
                if match:
                    train_loss = float(match.group(1))
                    train_loss_item = {"type": "train_loss", "value": float(train_loss)}
                    batches_loss.append(train_loss_item)
            elif "val_loss" in line:
                match = re.search(r"val_loss\s*=\s*([0-9.]+)", line)
                if match:
                    val_loss = float(match.group(1))
                    val_loss_item = {"type": "val_loss", "value": float(val_loss)}
                    batches_loss.append(val_loss_item)

        # Prepare data
        types = [item["type"] for item in batches_loss]
        values = [item["value"] for item in batches_loss]

        # Assign a color to each type
        type_colors = {
            "train_loss": "red",
            "val_loss": "green"
        }

        # Create a new figure
        plt.figure(figsize=(12, 6))

        # Plot segments with color changes
        for i in range(len(batches_loss) - 1):
            t1, t2 = types[i], types[i + 1]
            v1, v2 = values[i], values[i + 1]
            
            # If type changes, use the color of the starting point
            color = type_colors[t1]
            
            plt.plot([i, i + 1], [v1, v2], color=color, linewidth=2)

        # Add labels and grid
        plt.plot([1, 2, 3], [4, 5, 6])
        plt.title(f"Model Loss Over Batches\nModel GUID:{runtime_uuid}")        
        plt.xlabel("Batch")
        plt.ylabel("Loss")
        plt.grid(True)
        # Save the figure
        plt.savefig(f"{charts_path}/chart_{runtime_uuid}.png", bbox_inches='tight')
        # Show the figure
        plt.show()
        # Close the figure to free memory
        plt.close()

In [ ]:
# === Run the script ===
folder_path = f"{runtime_path}/outputs"
file_prefix = f"report"
_ = get_directory_files(folder_path, file_prefix)
process_chart_file_components(folder_path, file_prefix, runtime_uuid)